# K3: vektori reči + SVM + SMOTE (tvitovi avio-kompanijama)
**Tip:** Blok 3. Reprezentacija: vektori reči iz `en_core_web_md` (prosek vektora reči). Klasifikator: SVM. Balansiranje: SMOTE.

### Zadatak
U fajlu `Tweets.csv` su tvitovi putnika upućeni američkim avio-kompanijama, sa ručno označenim sentimentom.
- Od kolone `airline_sentiment` napraviti izlaznu varijablu `sentiment` (negativan, neutralan, pozitivan).
- Transformisati kolonu `text` primenom **vektora reči iz spaCy medium modela**. Tvit predstaviti kao prosek vektora reči posle čišćenja (izbaciti i pominjanja korisnika sa `@`).
- Prvi model: **SVC(kernel='linear', C=1)** uz **SMOTE** između vektorizacije i klasifikatora (imblearn Pipeline).
- Metrike po klasama i ukupno, balanced accuracy, matrica konfuzije. Interpretirati: koja klasa je najslabija i sa kojom se meša.
- Drugi model: komentar sa opsezima, pa podešavanje **C i kernel** (vektori reči nemaju hiperparametre transformacije) kroz 5-fold kros-validaciju.
- Uporediti `best_score_` sa macro F1 na testu, pa uporediti modele.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import RobustScaler, OneHotEncoder, MinMaxScaler, FunctionTransformer
from sklearn import metrics
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, balanced_accuracy_score
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline   # imblearn Pipeline radi i bez SMOTE-a, pa moze da ostane uvek

from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier

import matplotlib.pyplot as plt
import seaborn as sb

from pathlib import Path
from IPython.display import display

import spacy

RAND_STATE = 1
DATA_DIR = Path.cwd().parent / 'data'

### Šta je urađeno
- Učitani podaci, izbačeni tvitovi bez teksta i duplikati teksta (ne `dropna()` bez argumenata, jer kolone kao `negativereason` su prazne kod svih pozitivnih i neutralnih tvitova).
- Uzet uzorak od 4000 tvitova (`random_state=1`) zbog brzine.
- Napravljena izlazna varijabla `sentiment` (negativnih je oko 63%).
- Pregledane ostale kolone: `retweet_count` (kdeplot), `airline`, `tweet_location`, `user_timezone` (broj vrednosti i nedostajućih).
- Odluka: ulaz je samo `text`. `negativereason` i `airline_sentiment_confidence` bi bile curenje podataka (nastaju tek kad se sentiment označi).
- Podela 80:20 sa `stratify`.

In [ ]:
data = pd.read_csv(DATA_DIR / 'Tweets.csv')
data.info()

In [ ]:
data = data.dropna(subset=['text'])
print('Duplikati teksta:', data.duplicated(subset='text').sum())
data = data.drop_duplicates(subset='text')
data = data.sample(n=4000, random_state=RAND_STATE).reset_index(drop=True)
print(data.shape)

In [ ]:
sentiment_map = {'negative': 'negativan', 'neutral': 'neutralan', 'positive': 'pozitivan'}
data['sentiment'] = data['airline_sentiment'].map(sentiment_map)
print(data['sentiment'].value_counts())
print(data['sentiment'].value_counts(normalize=True))

In [ ]:
plt.figure(figsize=(10, 6))
sb.kdeplot(data=data, x='retweet_count', hue='sentiment',
           fill=True, common_norm=False, alpha=0.5, linewidth=2)
plt.title('Broj retvitova po sentimentu')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

print(data['retweet_count'].value_counts().head())
print(pd.DataFrame({'broj_vrednosti': data[['airline', 'tweet_location', 'user_timezone']].nunique(),
                    'nedostaje': data[['airline', 'tweet_location', 'user_timezone']].isna().sum()}))

In [ ]:
X = data['text']
y = data['sentiment']
y_encodings, y_levels = pd.factorize(y)
print(list(y_levels))

X_train, X_test, y_train, y_test = train_test_split(X, y_encodings,
                                                    test_size=0.2,
                                                    random_state=RAND_STATE,
                                                    stratify=y_encodings)
print(X_train.shape, X_test.shape)

---
# ▶ OVDE KREĆEŠ

Sve iznad je gotovo. Od ove ćelije pa nadalje kucaš sam: kod, komentare u markdown ćelijama i tumačenja.

**Tvoji koraci:**

1. `nlp_pipe = spacy.load('en_core_web_md', disable=['parser','ner'])` (sm nema vektore)
2. `spacy_srediTekst`: zadrži `not` i `no`, izbaci tokene koji počinju sa `@`, URL-ove, mejlove, brojeve
3. `tekst_u_vektore(tekstovi)` vraća `np.array` (broj tvitova x 300), pa `FunctionTransformer(tekst_u_vektore)`
4. `SMOTE(random_state=RAND_STATE)` i `SVC(kernel='linear', C=1, random_state=RAND_STATE, decision_function_shape='ovo')` u imblearn `Pipeline`
5. Metrike, balanced accuracy, matrica konfuzije, tumačenje
6. Komentar o hiperparametrima, `GridSearchCV` za `C` i `kernel` (savet: vektore izračunaj jednom, pa u grid ide samo SMOTE + SVC)
7. `best_score_` naspram testa, poređenje modela

---
## Dati kod
Ovo dobijaš uz zadatak. Pokreni ćeliju kad stigneš do poređenja modela.

In [ ]:
# DATI KOD (dobija se na kolokvijumu): poredjenje modela
def get_model_metrics(y_true: np.ndarray, y_pred: np.ndarray, model_name: str):
    report = classification_report(y_true, y_pred, output_dict=True)
    macro_avg = report['macro avg']
    weighted_avg = report['weighted avg']
    return {
        'Model': model_name,
        'Macro Precision': macro_avg['precision'],
        'Macro Recall': macro_avg['recall'],
        'Macro F1-Score': macro_avg['f1-score'],
        'Weighted Precision': weighted_avg['precision'],
        'Weighted Recall': weighted_avg['recall'],
        'Weighted F1-Score': weighted_avg['f1-score'],
        'Balanced Accuracy': balanced_accuracy_score(y_true, y_pred)
    }